# DPF — Distributed Point Function (BGI16)

ELL_IN ∈ [13, 31], ELL_OUT ∈ [2, 6].  Star topology: one Dealer,
two Evaluators.  Based on Boyle, Gilboa, and Ishai (CCS 2016).

``DpfDealer(ell_in, ell_out)`` returns a **type**; construct it
with ``(ch_eval0, ch_eval1)`` — channels to Evaluator 0 and 1.

``DpfEvaluator(ell_in, ell_out, party)`` returns a **type**;
construct it with ``(ch_dealer)`` — a channel to the Dealer.
*party* is 0 or 1.

In [ ]:
# This cell requires multi-party setup and will not run in a notebook

import mpmt
from mpmt.channels import wrap_socket, connect_retry

# Get types
DealerT = mpmt.DpfDealer(ell_in=20, ell_out=4)
Eval0T = mpmt.DpfEvaluator(ell_in=20, ell_out=4, party=0)
Eval1T = mpmt.DpfEvaluator(ell_in=20, ell_out=4, party=1)

# Build channels and inject — star topology
ch0 = wrap_socket(...)   # Dealer → Evaluator 0
ch1 = wrap_socket(...)   # Dealer → Evaluator 1
ch_d0 = connect_retry(host=..., port=...)  # Evaluator 0 → Dealer
ch_d1 = connect_retry(host=..., port=...)  # Evaluator 1 → Dealer

dealer = DealerT(ch0, ch1)
eval0 = Eval0T(ch_d0)
eval1 = Eval1T(ch_d1)

## Dealer: gen

``dealer.gen(alpha, beta)`` generates two DPF keys for a point
function f(x) = beta if x == alpha, else 0.  Returns ``(key0_json, key1_json)``.

*alpha* is the secret point (in Z_{2^{ell_in}}).  *beta* is the
output value (in Z_{2^{ell_out}}).

In [ ]:
# This cell requires multi-party setup and will not run in a notebook

key_e0, key_e1 = dealer.gen(alpha=42, beta=1)
# send key_e0 to Evaluator 0 via send_key, or application-layer transport

## Dealer: send_key

``dealer.send_key(key_json, party)`` sends a generated key to an
Evaluator over the injected channel.  *party* is 0 or 1.

``evaluator.recv_key()`` receives the key from the Dealer, returning
it as a JSON string.  No application-layer serialisation needed —
the channel handles it.

In [ ]:
# This cell requires multi-party setup and will not run in a notebook

# Dealer sends each key over the injected channel
dealer.send_key(key_e0, party=0)
dealer.send_key(key_e1, party=1)

# Each Evaluator receives its key
key_e0 = eval0.recv_key()
key_e1 = eval1.recv_key()

## Evaluator: eval (full domain)

``evaluator.eval(key_json, buf, cores=1)`` evaluates the DPF over the
entire domain [0, 2^{ell_in}).  *buf* is a pre-allocated ``RvectorPack``
of ``bf_size`` elements.  Results are written into *buf*.

The two evaluators' results are 2-of-2 additive shares of the point
function: ``eval0[i] + eval1[i] = f(i)``.

In [ ]:
# This cell requires multi-party setup and will not run in a notebook

buf = mpmt.RvectorPack(ell=4)(bf_size)
eval0.eval(key_json=key_e0, buf=buf, cores=1)

## Evaluator: eval_range

``evaluator.eval_range(key_json, buf, bg, ed, cores=1)`` evaluates
over a closed interval **[bg, ed]** (inclusive both ends).  Supports
circular intervals: when ``ed < bg`` the range wraps as
``[bg, 2^{ell_in}) ∪ [0, ed]``, with output linearised so *bg*
lands at offset 0.  Sub-trees outside the range are pruned.

## GenBF Protocol Usage

In the query protocol, the DPF is used to generate the query-side
Bloom filter without revealing the hash indices:

1. **Leader** (acting as S3 / Dealer) generates DPF keys for the
   **blinded** index ``idx_L + idx_A``, where ``idx_L`` is the ADD2
   hash share held by the Leader and ``idx_A`` is a random offset.
   Keys are sent to HelperA and HelperB.

2. **HelperA** and **HelperB** evaluate their keys over [0, bf_size).
   They each apply a **cyclic shift** by their RSS3 share of the
   blinding offset, producing 2-of-2 additive shares of the query BF.

3. **crng** + **reshare**: the 2-of-2 shares are converted to Rep3
   (2-of-3) via correlated randomness and a reshare round, so the
   three servers can compute the dot product.


## Channel Topology

DPF uses a **star** topology (not a ring):
```
        Dealer
       /      \
  Eval0      Eval1
```

Dealer listens on two ports; each Evaluator connects to one.
Channels are built with ``channels.wrap_socket`` and
``channels.connect_retry``, then injected at construction time.

## Properties

- ``dealer.ell_in``, ``dealer.ell_out`` — ring sizes
- ``evaluator.ell_in``, ``evaluator.ell_out``, ``evaluator.party``
